In [200]:
import os
import warnings
import pickle
import logging
import random
from copy import deepcopy as dc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

In [201]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

In [202]:
DATA_PATH       = "/kaggle/input/datasets/quanghuylt/stock-price/stock_price.csv"
METRICS_DIR     = "/kaggle/working/reports/metrics/lstm/"
RESULTS_DIR     = "/kaggle/working/data/meta_test/"
FIGURES_DIR     = "/kaggle/working/reports/figures/lstm"
MODEL_DIR       = "/kaggle/working/models/lstm/"
OOF_DIR         = "/kaggle/working/data/meta_train/"
SUMMARY_METRICS = "/kaggle/working/reports/metrics/lstm/LSTM_all_tickers.csv"

SYMBOLS      = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
FEATURE_COLS = ["Open", "High", "Low", "Close", "Volume"]
TARGET_COL   = "Close"

TRAIN_RATIO  = 0.8
N_FOLDS      = 10
GAP_DAYS     = 10
LOOKBACK     = 7

HIDDEN_SIZE  = 32
NUM_LAYERS   = 2
BATCH_SIZE   = 16
EPOCHS       = 50
LR           = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [203]:
def split_data(df: pd.DataFrame):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    split_idx = int(len(df) * TRAIN_RATIO)
    train_df  = df.iloc[:split_idx].reset_index(drop=True)
    test_df   = df.iloc[split_idx:].reset_index(drop=True)
    return train_df, test_df

In [206]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0  = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(x.device)
        c0  = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

In [ ]:
def build_xy_local(df: pd.DataFrame, n_steps: int):
    prices = df[TARGET_COL].values
    n = len(prices)

    X, Y = [], []
    mus, sigmas = [], []

    for i in range(n - n_steps):
        # 7-day look back window
        window_x = prices[i : i + n_steps] 
        target_y = prices[i + n_steps]

        mu = np.mean(window_x)
        sigma = np.std(window_x) + 1e-8

        scaled_x = (window_x - mu) / sigma
        scaled_y = (target_y - mu) / sigma

        X.append(scaled_x)
        Y.append(scaled_y)
        mus.append(mu)
        sigmas.append(sigma)

    X_t = torch.tensor(np.array(X).reshape(-1, n_steps, 1), dtype=torch.float32)
    Y_t = torch.tensor(np.array(Y).reshape(-1, 1), dtype=torch.float32)
    
    return X_t, Y_t, np.array(mus), np.array(sigmas)

In [208]:
def train_model(X_t: torch.Tensor, Y_t: torch.Tensor) -> LSTMModel:
    dataset = TensorDataset(X_t, Y_t)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model     = LSTMModel(input_size=1, hidden_size=HIDDEN_SIZE,
                          num_layers=NUM_LAYERS, output_size=1).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0:
            log.info(f"      epoch {epoch+1}/{EPOCHS}  loss={loss.item():.6f}")

    return model

In [209]:
def predict(model: LSTMModel, X_t: torch.Tensor) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        preds = model(X_t.to(DEVICE)).cpu().numpy().flatten()
    return preds

In [ ]:
def run_oof(train_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    n         = len(train_df)
    fold_size = n // (N_FOLDS + 1)
    oof_preds = np.full(n, np.nan)

    log.info(f"    OOF: {N_FOLDS} folds, fold_size={fold_size}, gap={GAP_DAYS}, lookback={LOOKBACK}")

    for k in range(1, N_FOLDS + 1):
        train_end = k * fold_size
        val_start = train_end + GAP_DAYS
        val_end   = (k + 1) * fold_size if k < N_FOLDS else n

        if val_start + LOOKBACK >= val_end:
            continue

        fold_train = train_df.iloc[:train_end].copy()
        fold_val   = train_df.iloc[val_start:val_end].copy()

        try:
            X_tr, Y_tr, _, _ = build_xy_local(fold_train, LOOKBACK)
            model = train_model(X_tr, Y_tr)

            context = list(fold_train[TARGET_COL].values[-LOOKBACK:])
            fold_preds = []
            
            model.eval()
            with torch.no_grad():
                for i in range(len(fold_val)):
                    window = np.array(context[-LOOKBACK:])
                    mu = np.mean(window)
                    sigma = np.std(window) + 1e-8
                    
                    scaled_window = (window - mu) / sigma
                    x = torch.tensor(scaled_window.reshape(1, LOOKBACK, 1), dtype=torch.float32).to(DEVICE)
                    
                    y_scaled = model(x).cpu().numpy().flatten()[0]
                    y_real = y_scaled * sigma + mu
                    
                    fold_preds.append(y_real)
                    context.append(fold_val[TARGET_COL].values[i]) 
            
            fill_len = min(len(fold_preds), val_end - val_start)
            oof_preds[val_start:val_start + fill_len] = fold_preds[:fill_len]
            
            mae = np.mean(np.abs(np.array(fold_preds[:fill_len]) - fold_val[TARGET_COL].values[:fill_len]))
            log.info(f"      Fold {k}: train=[0:{train_end}]  val=[{val_start}:{val_end}]  MAE={mae:.4f}")

            del model
            torch.cuda.empty_cache()

        except Exception as e:
            log.warning(f"      Fold {k} failed: {e}")
            oof_preds[val_start:val_end] = np.nan

    nan_count = np.isnan(oof_preds).sum()
    print(f"  oof nan: {nan_count}/{n} ({nan_count/n:.1%}) — first ~{fold_size + GAP_DAYS} rows uncovered")

    return pd.DataFrame({
        "Ticker":               ticker,
        "Date":                 train_df["Date"].values,
        "Close":                train_df[TARGET_COL].values,
        "LSTM_Predicted_Close": oof_preds,
    })

In [ ]:
def predict_test(model: LSTMModel, train_df: pd.DataFrame, test_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    model.eval()

    context = list(train_df[TARGET_COL].values[-LOOKBACK:])
    preds_real = []

    with torch.no_grad():
        for i in range(len(test_df)):
            window = np.array(context[-LOOKBACK:])
            
            mu = np.mean(window)
            sigma = np.std(window) + 1e-8
            
            scaled_window = (window - mu) / sigma
            x = torch.tensor(scaled_window.reshape(1, LOOKBACK, 1), dtype=torch.float32).to(DEVICE)
            
            y_scaled = model(x).cpu().numpy().flatten()[0]
            
            y_real = y_scaled * sigma + mu
            preds_real.append(y_real)
            
            context.append(test_df[TARGET_COL].values[i]) 

    return pd.DataFrame({
        "Ticker":               ticker,
        "Date":                 test_df["Date"].values,
        "Close":                test_df[TARGET_COL].values,
        "LSTM_Predicted_Close": np.array(preds_real),
    })

In [213]:
def calc_metrics(
    df: pd.DataFrame,
    actual_col: str = "Close",
    pred_col:   str = "LSTM_Predicted_Close",
    ticker_col: str = "Ticker",
) -> pd.DataFrame:

    if ticker_col and ticker_col in df.columns and df[ticker_col].nunique() > 1:
        return (
            df.groupby(ticker_col)
              .apply(lambda x: calc_metrics(x, actual_col, pred_col, ticker_col=None))
              .reset_index()
        )

    df = df.copy()
    df = df.dropna(subset=[actual_col, pred_col])
    df["Prev_Actual"] = df[actual_col].shift(1)
    df["Prev_Pred"]   = df[pred_col].shift(1)
    df = df.dropna(subset=["Prev_Actual", "Prev_Pred"])

    y_true = df[actual_col]
    y_pred = df[pred_col]

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8)))
    r2   = r2_score(y_true, y_pred)

    actual_dir = np.sign(y_true.values - df["Prev_Actual"].values)
    pred_dir   = np.sign(y_pred.values - df["Prev_Pred"].values)
    da         = np.mean(actual_dir == pred_dir)

    mask  = actual_dir != 0
    tpa   = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan

    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "MSE": mse, "MAE": mae, "MAPE": mape, "RMSE": rmse,
        "R2": r2, "DA": da, "TPA": tpa, "V-RMSE": v_rmse,
    }])

In [214]:
def plot_results(df: pd.DataFrame, save_dir: str) -> None:
    ticker  = df["Ticker"].iloc[0]
    df_plot = df.dropna(subset=["LSTM_Predicted_Close"])

    os.makedirs(save_dir, exist_ok=True)
    sns.set(style="whitegrid")

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    axes[0].plot(df_plot["Date"], df_plot["Close"],
                 label="Actual Close", color="#1f77b4", linewidth=2)
    axes[0].plot(df_plot["Date"], df_plot["LSTM_Predicted_Close"],
                 label="LSTM Predicted", color="#d62728", linestyle="--", linewidth=1.5)
    axes[0].set_title(f"LSTM predicted results: {ticker}", fontsize=15)
    axes[0].set_xlabel("Date", fontsize=12)
    axes[0].set_ylabel("Price", fontsize=12)
    axes[0].legend(loc="best")
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis="x", rotation=30)

    error  = df_plot["Close"] - df_plot["LSTM_Predicted_Close"]
    colors = np.where(error >= 0, "#2ca02c", "#d62728")
    axes[1].bar(df_plot["Date"], error, color=colors, alpha=0.6, width=1.5)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"Prediction error: {ticker}", fontsize=13)
    axes[1].set_xlabel("Date", fontsize=12)
    axes[1].set_ylabel("Error (Actual - Predicted)", fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"{ticker}.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()
    log.info(f"    Plot saved: {save_path}")

In [ ]:
def train_one_ticker(ticker: str) -> pd.DataFrame:
    log.info(f"========== {ticker} ==========")

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)

    train_df, test_df = split_data(df)
    log.info(f"  Train: {len(train_df)} rows  Test: {len(test_df)} rows")

    oof_df = run_oof(train_df, ticker)

    log.info("    Training final model on full train set...")
    X_tr, Y_tr, train_mus, train_sigmas = build_xy_local(train_df, LOOKBACK)
    final_model = train_model(X_tr, Y_tr)

    test_result_df = predict_test(final_model, train_df, test_df, ticker)

    final_model.eval()
    with torch.no_grad():
        train_preds_scaled = final_model(X_tr.to(DEVICE)).cpu().numpy().flatten()
    
    train_preds_real = train_preds_scaled * train_sigmas + train_mus

    train_refit_df = pd.DataFrame({
        "Ticker": ticker,
        "Date":   train_df["Date"].values[LOOKBACK:],
        "Close":  train_df[TARGET_COL].values[LOOKBACK:], 
        "LSTM_Predicted_Close": train_preds_real,
    })
    
    refit_metrics = calc_metrics(train_refit_df)
    refit_metrics.insert(0, "Ticker", ticker)
    log.info(f"  Refit Train — MAE={refit_metrics['MAE'].iloc[0]:.2f} DA={refit_metrics['DA'].iloc[0]:.2%} R2={refit_metrics['R2'].iloc[0]:.4f}")

    metrics_df = calc_metrics(oof_df)
    metrics_df.insert(0, "Ticker", ticker)
    log.info(f"  OOF  — MAE={metrics_df['MAE'].iloc[0]:.2f} DA={metrics_df['DA'].iloc[0]:.2%} R2={metrics_df['R2'].iloc[0]:.4f}")

    test_metrics = calc_metrics(test_result_df)
    test_metrics.insert(0, "Ticker", ticker)
    log.info(f"  Test — MAE={test_metrics['MAE'].iloc[0]:.2f} DA={test_metrics['DA'].iloc[0]:.2%} R2={test_metrics['R2'].iloc[0]:.4f}")

    os.makedirs(METRICS_DIR, exist_ok=True)
    metrics_df.to_csv(os.path.join(METRICS_DIR, f"LSTM_{ticker}_metrics.csv"), index=False)

    os.makedirs(OOF_DIR, exist_ok=True)
    oof_df.to_csv(os.path.join(OOF_DIR, f"oof_lstm_{ticker}.csv"), index=False)

    os.makedirs(RESULTS_DIR, exist_ok=True)
    test_result_df.to_csv(os.path.join(RESULTS_DIR, f"LSTM_{ticker}_predictions.csv"), index=False)

    plot_results(test_result_df, save_dir=FIGURES_DIR)

    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, f"lstm_{ticker}.pkl"), "wb") as f:
        pickle.dump({
            "model_state":  final_model.state_dict(),
            "ticker":       ticker
        }, f)

    torch.cuda.empty_cache()
    log.info(f"  All outputs saved for {ticker}")
    return test_metrics

In [217]:
all_metrics = []

for ticker in SYMBOLS:
    try:
        m = train_one_ticker(ticker)
        all_metrics.append(m)
    except Exception as e:
        log.error(f"{ticker} FAILED: {e}")

if all_metrics:
    summary_df = pd.concat(all_metrics, ignore_index=True)
    os.makedirs(os.path.dirname(SUMMARY_METRICS) or ".", exist_ok=True)
    summary_df.to_csv(SUMMARY_METRICS, index=False)
    log.info(f"Summary saved: {SUMMARY_METRICS}")
    print("\n" + summary_df.to_string(index=False))

log.info("Done.")

11:04:23  INFO      ========== VNM ==========
11:04:23  INFO        Train: 3259 rows  Test: 815 rows
11:04:23  INFO          OOF: 10 folds, fold_size=296, gap=10, lookback=7
11:04:23  INFO            epoch 10/50  loss=0.137466
11:04:24  INFO            epoch 20/50  loss=0.040558
11:04:24  INFO            epoch 30/50  loss=0.001290
11:04:25  INFO            epoch 40/50  loss=18.059071
11:04:25  INFO            epoch 50/50  loss=14.720358
11:04:25  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.1418
11:04:26  INFO            epoch 10/50  loss=1.733417
11:04:27  INFO            epoch 20/50  loss=0.666111
11:04:28  INFO            epoch 30/50  loss=1.577279
11:04:29  INFO            epoch 40/50  loss=0.358557
11:04:30  INFO            epoch 50/50  loss=1.010229
11:04:30  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.2837
11:04:32  INFO            epoch 10/50  loss=1.157763
11:04:33  INFO            epoch 20/50  loss=9.148299
11:04:34  INFO            epoch 30/50 

  oof nan: 396/3259 (12.2%) — first ~306 rows uncovered


11:06:34  INFO            epoch 10/50  loss=1.253721
11:06:39  INFO            epoch 20/50  loss=1.402603
11:06:44  INFO            epoch 30/50  loss=6.074183
11:06:49  INFO            epoch 40/50  loss=2.056669
11:06:54  INFO            epoch 50/50  loss=2.139081
11:06:54  INFO        Refit Train — MAE=0.90 DA=40.42% R2=0.9968
11:06:54  INFO        OOF  — MAE=0.91 DA=41.33% R2=0.9962
11:06:54  INFO        Test — MAE=0.97 DA=43.12% R2=0.8679
11:06:56  INFO          Plot saved: /kaggle/working/reports/figures/lstm/VNM.png
11:06:56  INFO        All outputs saved for VNM
11:06:56  INFO      ========== FPT ==========
11:06:56  INFO        Train: 3259 rows  Test: 815 rows
11:06:56  INFO          OOF: 10 folds, fold_size=296, gap=10, lookback=7
11:06:57  INFO            epoch 10/50  loss=0.011235
11:06:57  INFO            epoch 20/50  loss=0.028339
11:06:58  INFO            epoch 30/50  loss=0.101521
11:06:58  INFO            epoch 40/50  loss=0.044846
11:06:58  INFO            epoch 50/50  

  oof nan: 396/3259 (12.2%) — first ~306 rows uncovered


11:09:07  INFO            epoch 10/50  loss=8.133163
11:09:12  INFO            epoch 20/50  loss=0.013649
11:09:17  INFO            epoch 30/50  loss=13.844703
11:09:21  INFO            epoch 40/50  loss=1.393442
11:09:26  INFO            epoch 50/50  loss=0.409486
11:09:27  INFO        Refit Train — MAE=0.19 DA=46.11% R2=0.9993
11:09:27  INFO        OOF  — MAE=0.22 DA=46.09% R2=0.9991
11:09:27  INFO        Test — MAE=1.19 DA=50.49% R2=0.9946
11:09:28  INFO          Plot saved: /kaggle/working/reports/figures/lstm/FPT.png
11:09:28  INFO        All outputs saved for FPT
11:09:28  INFO      ========== MSN ==========
11:09:28  INFO        Train: 3259 rows  Test: 815 rows
11:09:28  INFO          OOF: 10 folds, fold_size=296, gap=10, lookback=7
11:09:29  INFO            epoch 10/50  loss=3.031887
11:09:29  INFO            epoch 20/50  loss=0.177550
11:09:30  INFO            epoch 30/50  loss=0.046711
11:09:30  INFO            epoch 40/50  loss=0.928447
11:09:31  INFO            epoch 50/50 

...

  oof nan: 396/3259 (12.2%) — first ~306 rows uncovered


11:16:45  INFO            epoch 10/50  loss=4.402324
11:16:50  INFO            epoch 20/50  loss=1.597957
11:16:55  INFO            epoch 30/50  loss=3.224349
11:17:00  INFO            epoch 40/50  loss=2.200331
11:17:05  INFO            epoch 50/50  loss=0.282801
11:17:05  INFO        Refit Train — MAE=0.30 DA=42.88% R2=0.9990
11:17:06  INFO        OOF  — MAE=0.33 DA=42.03% R2=0.9988
11:17:06  INFO        Test — MAE=1.20 DA=49.39% R2=0.9970
11:17:07  INFO          Plot saved: /kaggle/working/reports/figures/lstm/VIC.png
11:17:07  INFO        All outputs saved for VIC
11:17:07  INFO      ========== HPG ==========
11:17:07  INFO        Train: 3259 rows  Test: 815 rows
11:17:07  INFO          OOF: 10 folds, fold_size=296, gap=10, lookback=7
11:17:08  INFO            epoch 10/50  loss=17.726076
11:17:08  INFO            epoch 20/50  loss=0.703575
11:17:08  INFO            epoch 30/50  loss=1.210873
11:17:09  INFO            epoch 40/50  loss=9.670044
11:17:09  INFO            epoch 50/50 

  oof nan: 396/3259 (12.2%) — first ~306 rows uncovered


11:19:18  INFO            epoch 10/50  loss=2.897444
11:19:23  INFO            epoch 20/50  loss=0.820700
11:19:28  INFO            epoch 30/50  loss=1.260280
11:19:33  INFO            epoch 40/50  loss=1.332857
11:19:38  INFO            epoch 50/50  loss=1.187046
11:19:38  INFO        Refit Train — MAE=0.16 DA=44.88% R2=0.9984
11:19:38  INFO        OOF  — MAE=0.20 DA=46.09% R2=0.9976
11:19:38  INFO        Test — MAE=0.34 DA=44.23% R2=0.9808
11:19:40  INFO          Plot saved: /kaggle/working/reports/figures/lstm/HPG.png
11:19:40  INFO        All outputs saved for HPG
11:19:40  INFO      Summary saved: /kaggle/working/reports/metrics/lstm/LSTM_all_tickers.csv
11:19:40  INFO      Done.



Ticker      MSE      MAE     MAPE     RMSE       R2       DA      TPA   V-RMSE
   VNM 2.012001 0.971062 0.015971 1.418450 0.867948 0.431204 0.464286 1.417846
   FPT 2.915041 1.187198 0.013049 1.707349 0.994619 0.504914 0.527599 1.707348
   MSN 4.275271 1.345011 0.018233 2.067673 0.914457 0.472973 0.509259 2.064413
   VCB 0.800809 0.588846 0.009880 0.894880 0.951756 0.468059 0.510040 0.894879
   VIC 6.602303 1.199402 0.017494 2.569495 0.997028 0.493857 0.554483 2.550758
   HPG 0.238506 0.344611 0.015404 0.488371 0.980790 0.442260 0.476821 0.485069
